# R09 · Power and partitioning - DVFS, big.LITTLE, idle states, hypervisors

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/05_realtime_linux/R09_power_and_partitioning_dvfs_big_little_idle_stat.ipynb)

**Track:** Real-time and Linux · **Skill axis (sheet):** Real-time and Linux · **Time:** about 50 min

**Skill Hub concepts:** big.LITTLE and frequency scaling; CPU hotplug and idle states; Hypervisor and partitioning

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each low-level task has an optional C/C++ cell after it. Same tests: write the cell, then uncomment the `lab.check(..., "file.c")` line under it. A check passes if either language passes.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("R09")

## Map: spending joules, and keeping workloads apart

**Dynamic power** of CMOS logic: $P_{dyn} = C\,V^2 f$. Energy for a job of $N$ cycles at one operating point:
$E_{dyn} = C\,V^2 N$ (frequency cancels, voltage does not). Add static (leakage) power while running and idle power
for the rest of the period:
$$E = \big(P_{static} + C V^2 f\big)\frac{N}{f} + P_{idle}\Big(T - \frac{N}{f}\Big)$$

| Strategy | Wins when |
|---|---|
| Race to idle (high f, then deep idle) | static power is high and idle power is low |
| Slow and steady (lowest f that meets the deadline) | voltage can drop a lot at low f (energy goes with $V^2$) |

**big.LITTLE / DynamIQ:** fast power-hungry cores plus slow efficient ones; the scheduler (Energy Aware Scheduling)
places each task on the cheapest core that still meets its needs. **cpufreq governors** (`schedutil`, `performance`)
pick the operating point; real-time threads usually pin `performance` to avoid frequency ramps.

**Idle states (cpuidle):** deeper states save more power but take longer to wake (**exit latency**) and only pay off
if the core stays idle longer than their **target residency**. A real-time core often disables deep states
(`/dev/cpu_dma_latency`, `cpuidle.off=1`). **CPU hotplug** takes a core fully offline (`echo 0 > .../cpuN/online`).

**Hypervisors and partitioning:** type 1 (Xen, Jailhouse, a safety-certified separation kernel) run directly on the
hardware; type 2 (KVM on a host OS, VirtualBox) run inside one. **Static partitioning** (Jailhouse, Xen dom0less) gives
each guest fixed cores, memory and devices, so a safety-rated RTOS and Linux share one chip without sharing time.

## Task 1: pick the energy-optimal operating point
`opps` = list of `(f_hz, volts, p_static_w)`. Return the `(f, V, P_static)` tuple with the least energy over the window
that still finishes the job (`cycles / f <= window`).

In [ ]:
def job_energy(opp, cycles, window, c_eff, p_idle):
    raise NotImplementedError  # TODO

def best_opp(opps, cycles, window, c_eff, p_idle):
    raise NotImplementedError  # TODO

lab.check("dvfs", job_energy, best_opp)

In [ ]:
%%writefile r09_dvfs.c
// C/C++ version of Task 1 (optional, same tests). Declarations: roboprep/c/include/r09.h
// Grade it with lab.check("dvfs", "r09_dvfs.c"). For C++, name the file .cpp in both places.
#include "r09.h"

double job_energy(const Opp *opp, double cycles, double window, double c_eff, double p_idle) {
    (void)opp; (void)cycles; (void)window; (void)c_eff; (void)p_idle;
    return 0; /* TODO: (static + dynamic) power while running, idle power for the rest of the window */
}

int best_opp(const Opp *opps, size_t n, double cycles, double window, double c_eff, double p_idle) {
    (void)opps; (void)n; (void)cycles; (void)window; (void)c_eff; (void)p_idle;
    return -1; /* TODO */
}

In [ ]:
# lab.check("dvfs", "r09_dvfs.c")  # uncomment after writing the C/C++ cell above

## Task 2: choose an idle state
Pick the lowest-power state whose target residency fits the predicted idle time **and** whose exit latency fits the
latency limit. If none fit, return the shallowest state (first in the list).

In [ ]:
def pick_idle(states, predicted_idle_us, latency_limit_us):
    raise NotImplementedError  # TODO

lab.check("idle", pick_idle)

In [ ]:
%%writefile r09_idle.c
// C/C++ version of Task 2 (optional, same tests). Declarations: roboprep/c/include/r09.h
// Grade it with lab.check("idle", "r09_idle.c"). For C++, name the file .cpp in both places.
#include "r09.h"

size_t pick_idle(const IdleState *states, size_t n, double predicted_idle_us, double latency_limit_us) {
    (void)states; (void)n; (void)predicted_idle_us; (void)latency_limit_us;
    return 0; /* TODO: lowest power among states with residency <= predicted idle and exit latency <= limit */
}

In [ ]:
# lab.check("idle", "r09_idle.c")  # uncomment after writing the C/C++ cell above

## Checkpoint: Math
**M1 (Apply).** From 1.0 V at 2 GHz to 0.8 V at 1 GHz. Ratio of new to old dynamic power?

**M2 (Apply).** Same change, same job (same cycle count). Ratio of new to old dynamic energy?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)

## Checkpoint: Concept
**K1 (Apply).** The 1 kHz control core keeps missing deadlines by about 100 us after idle periods. Likely cause and fix?
- a) cache misses; add more RAM
- b) the core enters a deep idle state with long exit latency; cap the latency through `/dev/cpu_dma_latency` (or disable deep states on that core) and pin its frequency
- c) the hypervisor is too slow

**K2 (Evaluate).** A robot computer must run a safety monitor (certified RTOS) and Ubuntu with ROS 2 on the same quad-core
SoC. Compare a type 1 static-partitioning hypervisor with running everything on Linux.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `r09.h`).
3. In `roboprep/checks/r09.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()